In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [69]:
import math 
class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size: int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.W_k = nn.Linear(self.emb_size, self.head_size)
        self.W_q = nn.Linear(self.emb_size, self.head_size)
        self.W_v = nn.Linear(self.emb_size, self.head_size)

        self.mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len))

    def forward(self, x: torch.Tensor):
        batch_size, seq_len, emb_size = x.shape
        key = self.W_k(x)
        query = self.W_q(x)
        value = self.W_v(x)

        attention_matrix : torch.Tensor = query @ key.transpose(1,2) / math.sqrt(self.head_size)

        mask = self.mask[:seq_len, :seq_len] == 0
        
        attention_matrix.masked_fill_(mask, float('-inf'))
        
        attention_matrix = torch.softmax(attention_matrix, 2) 
        
        return attention_matrix @ value
        

In [70]:
torch.manual_seed(42)
HeadAttention(3,4,5).forward(torch.Tensor([[[1,2,3], [4,5,6]], [[6,5,4], [3,2,1]]]))

tensor([[[0.4092, 1.6838, 0.4976, 1.9791],
         [0.4659, 1.8297, 0.5034, 2.0753]],

        [[2.7387, 4.2041, 0.4221, 2.4407],
         [2.3405, 3.1779, 0.3814, 1.7644]]], grad_fn=<UnsafeViewBackward0>)

In [62]:
ones= torch.tril(torch.ones(3,3,3))

In [12]:
ones.transpose(1,2)

tensor([[[1., 1., 1.],
         [0., 1., 1.],
         [0., 0., 1.]],

        [[1., 1., 1.],
         [0., 1., 1.],
         [0., 0., 1.]],

        [[1., 1., 1.],
         [0., 1., 1.],
         [0., 0., 1.]]])

In [30]:
ones[:,torch.BoolTensor([[True, False, False],[True, True, False],[True, True, True]])]

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [36]:
ones.masked_fill_(torch.BoolTensor([[True, False, False],[True, True, False],[True, True, True]]), float('-inf'))

tensor([[[-inf, 0., 0.],
         [-inf, -inf, 0.],
         [-inf, -inf, -inf]],

        [[-inf, 0., 0.],
         [-inf, -inf, 0.],
         [-inf, -inf, -inf]],

        [[-inf, 0., 0.],
         [-inf, -inf, 0.],
         [-inf, -inf, -inf]]])